# Train and Export the RLHF Playground cVAE

This notebook runs the same training/export scripts as the local backend commands. It is meant for Google Colab, but it also works in Jupyter if you run it from the repo root.

Expected outputs:

- `backend_training/checkpoints/cvae.pt`
- `backend_training/checkpoints/recon_preview.png`
- `backend_training/checkpoints/sample_grid_digit7.png`
- `public/models/decoder.onnx`
- `public/models/decoder.onnx.data`
- `public/models/metadata.json`


## 1. Runtime Setup

In Colab, choose **Runtime > Change runtime type > T4 GPU** or another GPU runtime before running this notebook.

In [ ]:
from pathlib import Path
import os
import sys

# If this notebook is inside the repo root, this is already correct.
# In Colab, set this to your copied/cloned repo directory if needed.
REPO_DIR = Path.cwd()
os.chdir(REPO_DIR)

print('Working directory:', Path.cwd())
print('Python:', sys.version)


## 2. Install Python Dependencies

Colab usually already has PyTorch, but this keeps the export dependencies available too.

In [ ]:
%pip install -q torch torchvision onnx onnxscript

## 3. Verify Repo Files

These checks make sure the notebook is using the same implementation as the local commands.

In [ ]:
required_files = [
    Path('backend_training/data/colored_mnist.py'),
    Path('backend_training/models/cvae.py'),
    Path('backend_training/train_cvae.py'),
    Path('backend_training/export_decoder.py'),
    Path('backend_training/sample_grid.py'),
]

missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError('Missing repo files: ' + ', '.join(missing))

print('All backend training files found.')


## 4. Train the cVAE

This uses the exact same script as local training. The defaults are `latent_dim=2` and `image_size=64`.

For a quick smoke test, set `EPOCHS = 1`. For a usable demo, start with `EPOCHS = 20`.

In [ ]:
import torch

EPOCHS = 20
BATCH_SIZE = 256
NUM_WORKERS = 2
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Device:', DEVICE)
!python backend_training/train_cvae.py --epochs {EPOCHS} --batch-size {BATCH_SIZE} --num-workers {NUM_WORKERS} --device {DEVICE}


## 5. Export the Decoder for the Website

This writes the same frontend files used by the Vite app.

In [ ]:
!python backend_training/export_decoder.py

## 6. Sample a Latent Grid for Visual Inspection

This image is just for checking decoder quality before you use the website.

In [ ]:
SAMPLE_DIGIT = 7
!python backend_training/sample_grid.py --digit {SAMPLE_DIGIT} --out backend_training/checkpoints/sample_grid_digit{SAMPLE_DIGIT}.png --device {DEVICE}


In [ ]:
from IPython.display import Image, display

preview = Path(f'backend_training/checkpoints/sample_grid_digit{SAMPLE_DIGIT}.png')
if preview.exists():
    display(Image(filename=str(preview)))
else:
    print('Preview image not found:', preview)


## 7. Confirm Output Files

Copy `public/models/decoder.onnx`, `public/models/decoder.onnx.data`, and `public/models/metadata.json` back into the local repo if you trained in Colab.

In [ ]:
outputs = [
    Path('backend_training/checkpoints/cvae.pt'),
    Path('backend_training/checkpoints/recon_preview.png'),
    Path(f'backend_training/checkpoints/sample_grid_digit{SAMPLE_DIGIT}.png'),
    Path('public/models/decoder.onnx'),
    Path('public/models/decoder.onnx.data'),
    Path('public/models/metadata.json'),
]

for path in outputs:
    status = 'OK' if path.exists() else 'MISSING'
    size = path.stat().st_size if path.exists() else 0
    print(f'{status:7} {size:>12} bytes  {path}')


## Optional: Zip the Website Model Files

This makes it easier to download the exact files needed by the frontend.

In [ ]:
!zip -j public/models/frontend_decoder_assets.zip public/models/decoder.onnx public/models/decoder.onnx.data public/models/metadata.json
print('Created public/models/frontend_decoder_assets.zip')
